
# ASL Fingerspelling — BERT-style Encoder + CTC (PyTorch Lightning)

**Competition:** Google - American Sign Language Fingerspelling Recognition  
**Link:** https://www.kaggle.com/competitions/asl-fingerspelling

This notebook trains a lightweight BERT-style Transformer encoder over MediaPipe landmark sequences and decodes characters with **CTC loss**.  
It's kept compact and readable so you can extend it.

**Key ideas**
- Parse the `.parquet` landmark sequences, auto-pick useful feature columns (hands + pose if available).
- Normalize per-sequence; simple frame-drop augmentation optional.
- BERT-ish encoder: Linear projection → Positional embeddings → TransformerEncoder (multi-head self-attn).
- Per-frame logits over characters + CTC blank; greedy decode for inference.
- Makes a valid `submission.csv`.

> Tip: Start with a small subset (`N_TRAIN_ROWS`) to verify everything runs, then scale up.


In [1]:

# Install/Imports (Kaggle usually has these, but this is safe)
# If you hit version issues, you can pin versions here.
# %pip -q install --upgrade pip
# %pip -q install pytorch-lightning==2.2.5 transformers==4.41.2 pyarrow==15.0.2

import os
import math
import time
import random
import gc
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

try:
    import lightning as L   # newer package name
except Exception:
    import pytorch_lightning as L

from typing import List, Dict, Tuple

print('torch', torch.__version__)
try:
    import lightning
    print('lightning', lightning.__version__)
except Exception as e:
    print('lightning import fallback ok')
    
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device


torch 2.6.0+cu124
lightning import fallback ok


'cuda'

In [2]:

# Config
class CFG:
    INPUT_DIR = Path('/kaggle/input/asl-fingerspelling')
    TRAIN_CSV = INPUT_DIR / 'train.csv'          # expected
    SUPP_CSV  = INPUT_DIR / 'supplemental_metadata.csv'  # optional (exists in competition)
    SAMPLE_SUB = INPUT_DIR / 'sample_submission.csv'     # for column names
    LANDMARKS_ROOTS = [
        INPUT_DIR / 'train_landmarks', 
        INPUT_DIR / 'supplemental_landmarks',  # some setups have extra data here
        INPUT_DIR
    ]

    # Training
    EPOCHS = 3
    BATCH_SIZE = 12
    LR = 1e-3
    WD = 1e-4
    NUM_WORKERS = 2
    MAX_TOKENS_PER_BATCH = 12000   # dynamic batching cap (seq_len * batch)
    GRAD_ACCUM = 1

    # Model
    D_MODEL = 256
    N_HEADS = 8
    N_LAYERS = 6
    DROPOUT = 0.1
    FFN_MULT = 4

    # Limits for quick sanity-runs; set to None to use all
    N_TRAIN_ROWS = 8000   # None for full dataset
    N_VAL_ROWS = 1500
    MAX_FRAMES = 256      # cap sequence length for speed (keep most informative frames)

    # Aug
    FRAME_DROP_P = 0.05

cfg = CFG()
cfg


In [3]:

# Utility: locate parquet by 'path' value in CSV.
def resolve_parquet_path(rel_path: str) -> Path:
    # CSV 'path' often lacks extension; ensure .parquet
    cand = Path(rel_path)
    if cand.suffix != '.parquet':
        cand = cand.with_suffix('.parquet')
    # Try typical roots
    for root in cfg.LANDMARKS_ROOTS:
        p = root / cand.name if root.is_dir() else root / cand
        if p.exists():
            return p
    # Fallback: search
    for root in cfg.LANDMARKS_ROOTS:
        if root.is_dir():
            q = list(root.rglob(cand.name))
            if q:
                return q[0]
    return cand

# Read training CSV
train_df = pd.read_csv(cfg.TRAIN_CSV)
if cfg.N_TRAIN_ROWS is not None and len(train_df) > cfg.N_TRAIN_ROWS:
    train_df = train_df.sample(cfg.N_TRAIN_ROWS, random_state=SEED).reset_index(drop=True)
train_df.head(3)


,path,file_id,sequence_id,participant_id,phrase
0,train_landmarks/1365275733.parquet,1365275733,2092898448,70,supremasi
1,train_landmarks/2072876091.parquet,2072876091,1945618977,59,100-407-5928
2,train_landmarks/2118949241.parquet,2118949241,387033856,25,8 joshua simmons drive


In [4]:

# Character vocabulary from training phrases
def build_vocab(phrases: pd.Series) -> Tuple[Dict[str,int], Dict[int,str]]:
    chars = sorted(set(''.join(list(phrases.astype(str).values))))
    # ensure space and apostrophe are present
    for ch in [' ', "'", '-', '.', ',', '/', '&', '#', '!', '?']:
        if ch in chars: 
            pass
    # Create mapping
    stoi = {ch:i for i,ch in enumerate(chars, start=0)}  # 0..K-1 are characters
    # CTC requires a blank token id; we'll add it as the last index
    blank_id = len(stoi)
    itos = {i:ch for ch,i in stoi.items()}
    return stoi, itos, blank_id

stoi, itos, CTC_BLANK = build_vocab(train_df['phrase'])
len(stoi), CTC_BLANK


(53, 53)

In [5]:

# Identify useful landmark feature columns dynamically from a sample parquet.
# We prioritize hands; include pose and (optionally) selected face keypoints to limit dims.
import pyarrow.parquet as pq

def probe_feature_columns(sample_parquet_path: Path, max_face=0, include_pose=True, include_hands=True) -> List[str]:
    table = pq.read_table(sample_parquet_path, columns=None, memory_map=True)
    cols = list(map(str, table.column_names))
    selected = []

    def add_if(patterns):
        for c in cols:
            ok = True
            for pat in patterns:
                if pat not in c:
                    ok = False; break
            if ok:
                selected.append(c)

    if include_hands:
        # common patterns across kernels: 'left_hand', 'right_hand'
        add_if(['left_hand'])
        add_if(['right_hand'])

    if include_pose:
        add_if(['pose'])

    # Face can be huge; optionally take a small, consistent subset
    if max_face > 0:
        face_cols = [c for c in cols if 'face' in c]
        face_cols = sorted(face_cols)[:max_face]
        selected.extend(face_cols)

    # Ensure unique
    selected = sorted(list(set(selected)))
    return selected

sample_path = resolve_parquet_path(train_df.loc[0, 'path'])
feature_cols = probe_feature_columns(sample_path, max_face=0, include_pose=True, include_hands=True)
print('Num features (columns):', len(feature_cols))
feature_cols[:10]


Num features (columns): 225


['x_left_hand_0',
 'x_left_hand_1',
 'x_left_hand_10',
 'x_left_hand_11',
 'x_left_hand_12',
 'x_left_hand_13',
 'x_left_hand_14',
 'x_left_hand_15',
 'x_left_hand_16',
 'x_left_hand_17']

In [6]:

# Dataset
class ASLDataset(Dataset):
    def __init__(self, df: pd.DataFrame, feature_cols: List[str], phase='train'):
        self.df = df.reset_index(drop=True)
        self.feature_cols = feature_cols
        self.phase = phase

    def __len__(self): return len(self.df)

    def load_parquet(self, p: str) -> pd.DataFrame:
        fpath = resolve_parquet_path(p)
        df = pd.read_parquet(fpath)
        # Keep only selected features if present; otherwise take numeric cols
        cols = [c for c in self.feature_cols if c in df.columns]
        if len(cols) == 0:
            cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
        df = df[cols].copy()
        return df

    def seq_augment(self, x: np.ndarray) -> np.ndarray:
        if self.phase == 'train' and cfg.FRAME_DROP_P > 0:
            mask = np.random.rand(len(x)) > cfg.FRAME_DROP_P
            if mask.sum() < 4:
                mask[np.random.choice(len(x), 4, replace=False)] = True
            x = x[mask]
        return x

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        phrase = str(row['phrase'])
        # Load landmarks
        df_lm = self.load_parquet(row['path'])
        x = df_lm.values.astype('float32')
        # Replace inf/nan
        x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
        # Normalize per-sequence
        mean = x.mean(axis=0, keepdims=True)
        std  = x.std(axis=0, keepdims=True) + 1e-6
        x = (x - mean) / std

        # Cap length for speed
        if cfg.MAX_FRAMES is not None and len(x) > cfg.MAX_FRAMES:
            # Keep middle chunk (often most informative)
            start = max(0, (len(x) - cfg.MAX_FRAMES)//2)
            x = x[start:start+cfg.MAX_FRAMES]
        x = self.seq_augment(x)

        # Targets
        target = np.array([stoi[ch] for ch in phrase], dtype=np.int64)

        return {
            'x': torch.from_numpy(x),                # [T, F]
            'y': torch.from_numpy(target),           # [U]
            'path': row['path'],
            'phrase': phrase,
        }

def ctc_collate(batch):
    # dynamic padding
    xs = [b['x'] for b in batch]
    ys = [b['y'] for b in batch]
    paths = [b['path'] for b in batch]
    phrases = [b['phrase'] for b in batch]

    T = [x.size(0) for x in xs]
    U = [y.size(0) for y in ys]
    Fdim = xs[0].size(1)

    max_T = max(T)
    max_U = max(U)

    pad_x = torch.zeros(len(xs), max_T, Fdim, dtype=xs[0].dtype)
    for i,x in enumerate(xs):
        pad_x[i,:x.size(0)] = x

    pad_y = torch.full((len(ys), max_U), fill_value=-1, dtype=torch.long)
    for i,y in enumerate(ys):
        pad_y[i,:y.size(0)] = y

    return {
        'x': pad_x,
        'x_len': torch.tensor(T, dtype=torch.long),
        'y': pad_y,
        'y_len': torch.tensor(U, dtype=torch.long),
        'paths': paths,
        'phrases': phrases,
    }


In [7]:

# Train/val split (stratify on first char if possible to keep it simple)
if cfg.N_VAL_ROWS:
    val_df = train_df.sample(cfg.N_VAL_ROWS, random_state=SEED)
    tr_df  = train_df.drop(val_df.index).reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)
else:
    tr_df = train_df.copy()
    val_df = train_df.iloc[:0].copy()

train_ds = ASLDataset(tr_df, feature_cols, 'train')
val_ds   = ASLDataset(val_df, feature_cols, 'val')

def make_loader(ds, shuffle, batch_size=None):
    if batch_size is None:
        batch_size = cfg.BATCH_SIZE
    return DataLoader(
        ds, shuffle=shuffle, batch_size=batch_size, num_workers=cfg.NUM_WORKERS,
        collate_fn=ctc_collate, pin_memory=True, drop_last=False
    )

train_loader = make_loader(train_ds, True)
val_loader   = make_loader(val_ds, False)

len(train_ds), len(val_ds)


(6500, 1500)

In [8]:

# BERT-style Encoder with positional embeddings + TransformerEncoder
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=4096):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe, persistent=False)

    def forward(self, x):
        # x: [B, T, D]
        T = x.size(1)
        return x + self.pe[:, :T]

class BertCTC(nn.Module):
    def __init__(self, in_features, vocab_size, d_model=256, n_layers=6, n_heads=8, dropout=0.1, ffn_mult=4, blank_id=None):
        super().__init__()
        self.proj = nn.Linear(in_features, d_model)
        self.posenc = PositionalEncoding(d_model)
        enc_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, dim_feedforward=d_model*ffn_mult, dropout=dropout, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(d_model, vocab_size + 1)  # +1 for CTC blank at the end by convention
        self.blank_id = vocab_size if blank_id is None else blank_id

    def forward(self, x, key_padding_mask=None):
        # x: [B, T, F]
        x = self.proj(x)
        x = self.posenc(x)
        x = self.encoder(x, src_key_padding_mask=key_padding_mask)
        x = self.dropout(x)
        logits = self.head(x)  # [B, T, V+1]
        return logits


In [9]:

# Lightning Module
class LitCTC(L.LightningModule):
    def __init__(self, in_features, vocab_size, blank_id):
        super().__init__()
        self.save_hyperparameters(ignore=['in_features'])
        self.model = BertCTC(
            in_features=in_features, vocab_size=vocab_size,
            d_model=cfg.D_MODEL, n_layers=cfg.N_LAYERS, n_heads=cfg.N_HEADS,
            dropout=cfg.DROPOUT, ffn_mult=cfg.FFN_MULT, blank_id=blank_id
        )
        self.ctc = nn.CTCLoss(blank=blank_id, zero_infinity=True)

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=cfg.LR, weight_decay=cfg.WD)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.EPOCHS)
        return {"optimizer": opt, "lr_scheduler": sch}

    @staticmethod
    def make_mask(x_len, max_T):
        # True means pad (masked)
        mask = torch.arange(max_T, device=x_len.device)[None, :].expand(len(x_len), max_T) >= x_len[:, None]
        return mask  # [B, T]

    def training_step(self, batch, batch_idx):
        x, x_len, y, y_len = batch['x'], batch['x_len'], batch['y'], batch['y_len']
        key_padding = self.make_mask(x_len, x.size(1))
        logits = self.model(x, key_padding_mask=key_padding)  # [B,T,V+1]
        logp = logits.log_softmax(-1)                         # CTC expects log-probs

        # Prepare targets for CTC (flatten and strip -1s)
        target = torch.cat([yy[yy!=-1] for yy in y], dim=0)
        # Input lengths and target lengths
        input_lengths = x_len
        target_lengths = y_len

        loss = self.ctc(logp.permute(1,0,2), input_lengths, target, target_lengths)  # time-major for CTC
        self.log('train_loss', loss, prog_bar=True, on_step=True, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, x_len, y, y_len = batch['x'], batch['x_len'], batch['y'], batch['y_len']
        key_padding = self.make_mask(x_len, x.size(1))
        logits = self.model(x, key_padding_mask=key_padding)
        logp = logits.log_softmax(-1)

        target = torch.cat([yy[yy!=-1] for yy in y], dim=0)
        input_lengths = x_len
        target_lengths = y_len

        loss = self.ctc(logp.permute(1,0,2), input_lengths, target, target_lengths)
        self.log('val_loss', loss, prog_bar=True)

        # quick greedy decode for a rough metric (CER proxy)
        with torch.no_grad():
            pred_ids = logp.argmax(-1)  # [B,T]
            preds = greedy_ctc_decode(pred_ids.cpu().numpy(), cfg_blank=CTC_BLANK)
            gts = [ ''.join(itos[i.item()] for i in yy[yy!=-1]) for yy in y ]
            cer = np.mean([char_error_rate(p, t) for p,t in zip(preds, gts)])
            self.log('val_cer', cer, prog_bar=True)

        return loss

def greedy_ctc_decode(pred_ids: np.ndarray, cfg_blank: int) -> List[str]:
    texts = []
    for seq in pred_ids:
        prev = -1
        chars = []
        for t in seq:
            t = int(t)
            if t == cfg_blank: 
                prev = -1
                continue
            if t != prev:
                chars.append(itos.get(t, '?'))
            prev = t
        texts.append(''.join(chars))
    return texts

def char_error_rate(pred: str, tgt: str) -> float:
    # basic Levenshtein
    m, n = len(pred), len(tgt)
    dp = np.zeros((m+1, n+1), dtype=np.int32)
    dp[:,0] = np.arange(m+1)
    dp[0,:] = np.arange(n+1)
    for i in range(1, m+1):
        for j in range(1, n+1):
            cost = 0 if pred[i-1] == tgt[j-1] else 1
            dp[i,j] = min(dp[i-1,j]+1, dp[i,j-1]+1, dp[i-1,j-1]+cost)
    return dp[m,n] / max(1, n)


In [10]:

# Instantiate model
# Infer input feature dim from a sample
sample_x = train_ds[0]['x']
in_features = sample_x.size(1)
vocab_size = len(stoi)
model = LitCTC(in_features, vocab_size, blank_id=CTC_BLANK)
print('in_features:', in_features, 'vocab_size:', vocab_size, 'blank_id:', CTC_BLANK)


in_features: 225 vocab_size: 53 blank_id: 53


In [ ]:

from pytorch_lightning.callbacks import ModelCheckpoint, LearningRateMonitor

# Trainer
callbacks = [
    ModelCheckpoint(
        monitor='val_cer',
        mode='min',
        save_top_k=1,
        filename='bertctc-{epoch:02d}-{val_cer:.4f}'
    ),
    LearningRateMonitor(logging_interval='step'),
]

trainer = L.Trainer(
    max_epochs=cfg.EPOCHS,
    gradient_clip_val=1.0,
    accumulate_grad_batches=cfg.GRAD_ACCUM,
    precision='16-mixed' if torch.cuda.is_available() else 32,
    devices=1,
    callbacks=callbacks,
    log_every_n_steps=20,
    enable_checkpointing=True,
)

trainer.fit(model, train_loader, val_loader)


2025-08-13 11:44:00.827946: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1755085440.856274     215 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1755085440.864702     215 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

In [ ]:

# Inference on test set and make submission.
test_csv = pd.read_csv(cfg.INPUT_DIR / 'test.csv')
print('Test rows:', len(test_csv))
test_paths = test_csv['path'].tolist()

test_pred_phrases = []

# Create a plain nn.Module for inference
infer_net = model.model.eval().to(device)

def run_one(pth: str) -> str:
    df = pd.read_parquet(resolve_parquet_path(pth))
    cols = [c for c in feature_cols if c in df.columns]
    if len(cols) == 0:
        cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    x = df[cols].values.astype('float32')
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    mean = x.mean(axis=0, keepdims=True); std = x.std(axis=0, keepdims=True) + 1e-6
    x = (x - mean) / std
    if cfg.MAX_FRAMES is not None and len(x) > cfg.MAX_FRAMES:
        start = max(0, (len(x) - cfg.MAX_FRAMES)//2)
        x = x[start:start+cfg.MAX_FRAMES]
    xt = torch.from_numpy(x).unsqueeze(0).to(device)  # [1,T,F]
    with torch.no_grad():
        logit = infer_net(xt)                 # [1,T,V+1]
        pred = logit.argmax(-1).squeeze(0).cpu().numpy()
        text = greedy_ctc_decode(np.expand_dims(pred,0), cfg_blank=CTC_BLANK)[0]
    return text

for i,p in enumerate(test_paths):
    text = run_one(p)
    test_pred_phrases.append(text if len(text)>0 else 'a')  # avoid empty

sub = pd.DataFrame({'path': test_paths, 'phrase': test_pred_phrases})
sub.to_csv('submission.csv', index=False)
sub.head()



## Notes & Next Steps

- This is a **baseline**; leaderboard score will be modest. To improve:
  - Train longer and on full data. Use `supplemental_metadata.csv` if available by merging its paths.
  - Add stronger augs: temporal jitters, dropout of random fingers (masking columns), CutMix on time axis.
  - Use a CTC-aware decoder with an external character-level LM (beam search).
  - Try Conformer or Squeezeformer encoder blocks.
  - Include a **length predictor** head and use alignment heuristics from top solutions.

- If columns don't match on your fork, print a parquet's `df.columns` and tweak `probe_feature_columns` filters.
